# Transparency and Explainability
## Module 5, Lesson 3

In this notebook, we will:
1. Train a Random Forest classifier on a loan dataset
2. Use LIME to explain individual predictions
3. Use SHAP to compute feature attributions
4. Compare global and local explanations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

np.random.seed(42)

## 1. Generate Dataset

We create a synthetic loan approval dataset with 5 features.

In [ ]:
n = 1000

income = np.random.normal(60, 25, n)
credit_score = np.random.normal(680, 60, n)
loan_amount = np.random.uniform(1000, 50000, n)
years_employed = np.random.uniform(0, 35, n)
debt_ratio = np.random.uniform(0.1, 0.8, n)

score = (credit_score - 600) / 200 * 0.5 + \
        (income - 40) / 80 * 0.3 + \
        (years_employed - 10) / 25 * 0.1 + \
        (0.4 - debt_ratio) * 0.3 + \
        np.random.normal(0, 0.1, n)

approved = (score > 0.5).astype(int)

df = pd.DataFrame({
    'income': income.round(1),
    'credit_score': credit_score.round(0),
    'loan_amount': loan_amount.round(0),
    'years_employed': years_employed.round(1),
    'debt_ratio': debt_ratio.round(3),
    'approved': approved
})

print("Dataset shape:", df.shape)
print("\nApproval rate:", df['approved'].mean())
print(df.head())

## 2. Train a Black Box Model (Random Forest)

In [ ]:
feature_cols = ['income', 'credit_score', 'loan_amount', 'years_employed', 'debt_ratio']
X = df[feature_cols]
y = df['approved']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Random Forest Accuracy:", (y_pred == y_test).mean())
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Global feature importance (inherent to Random Forest)
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 4))
plt.barh(importances['feature'], importances['importance'])
plt.xlabel('Feature Importance')
plt.title('Random Forest Global Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 3. LIME Explanations

We use LIME to explain individual predictions. Install with `pip install lime`.

In [ ]:
import lime
import lime.lime_tabular

explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train.values,
    feature_names=feature_cols,
    class_names=['denied', 'approved'],
    mode='classification',
    random_state=42
)

In [ ]:
idx = 0
instance = X_test.iloc[idx]

print("Instance to explain:")
print(instance)
print(f"\nActual label: {'approved' if y_test.iloc[idx] == 1 else 'denied'}")
print(f"Prediction: {'approved' if y_pred[idx] == 1 else 'denied'}")
print(f"Probability of approval: {rf.predict_proba(instance.values.reshape(1, -1))[0][1]:.3f}")

In [ ]:
exp = explainer.explain_instance(
    instance.values,
    rf.predict_proba,
    num_features=5
)

print("LIME Explanation (feature contributions):")
for feat, contrib in exp.as_list():
    sign = "+" if contrib > 0 else ""
    print(f"  {feat}: {sign}{contrib:.4f}")

exp.show_in_notebook(show_table=True)

## 4. SHAP Explanations

SHAP provides theoretically grounded feature attributions. Install with `pip install shap`.

In [ ]:
import shap

explainer_shap = shap.TreeExplainer(rf)
shap_values = explainer_shap.shap_values(X_test)

# Summary plot (global explanation)
shap.summary_plot(shap_values[1], X_test, feature_names=feature_cols, show=False)
plt.tight_layout()
plt.show()

In [ ]:
# Force plot for a single prediction (local explanation)
shap.force_plot(
    explainer_shap.expected_value[1],
    shap_values[1][idx, :],
    X_test.iloc[idx],
    feature_names=feature_cols,
    matplotlib=True
)
plt.tight_layout()
plt.show()

In [ ]:
# Waterfall plot for detailed breakdown
shap.plots.waterfall(shap.Explanation(
    values=shap_values[1][idx],
    base_values=explainer_shap.expected_value[1],
    data=X_test.iloc[idx].values,
    feature_names=feature_cols
), max_display=10)
plt.tight_layout()
plt.show()

## 5. Comparison: Interpretable Model

Train a logistic regression and compare feature coefficients (interpretable by nature) with SHAP values.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': lr.coef_[0]
}).sort_values('coefficient', ascending=False)

print("Logistic Regression Coefficients:")
print(coef_df)
print("\n(These are directly interpretable: positive = increases approval probability)")

## 6. Exercises

### Basic
1. Explain 3 different instances from the test set using LIME. How do the explanations differ?
2. Compare LIME and SHAP explanations for the same instance. Do they agree on feature importance rank?

### Implementation
3. Train a Gradient Boosting model and explain it with SHAP. Compare the explanations with the Random Forest.
4. Create a function that takes a model and an instance, and returns both LIME and SHAP explanations as DataFrames.

### Critical Thinking
5. LIME explanations can be unstable (different runs give different results). Design a small experiment to test LIME stability: run LIME 10 times on the same instance and measure the variance in feature importance.
6. Should post-hoc explanations be legally sufficient for high-stakes decisions like loan denial or medical diagnosis? What are the arguments for and against?